# 03 · Analyze representation geometry

 **Question:** How do the dimensions and similarity of internal representations change
 as a compiled transformer executes its algorithm?

 Read [01 · Programs](01_programs.ipynb) and [02 · Trace](02_trace.ipynb) for context.
 This notebook builds its own models and needs no earlier notebook state.

## Configure

 Use seed 42, length 6, and 64 target observations per reachable output class to
 reproduce the reference design. Balance classes separately for each program and
 distribute targets across reachable positions. Repeated targets retain their weight.

In [ ]:
import sys
from pathlib import Path

# Run from the repository root or any folder inside it.
start = Path.cwd().resolve()
PROJECT_ROOT = next(
    (root for root in (start, *start.parents) if (root / "raspformer").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Open this notebook from inside the RaspFormer repository.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

%matplotlib inline
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from raspformer.compiler import CompilerConfig
from raspformer.geometry import (
    ProbeConfig,
    lane_variances,
    pca_coordinates,
    representation,
    run_geometry_study,
    trajectory_coordinates,
)
from raspformer.programs import build_programs

plt.rcParams.update(
    {
        "figure.dpi": 110,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "font.size": 10,
    }
)

In [ ]:
compiler_config = CompilerConfig(
    vocab=tuple(range(-2, 6)),
    max_seq_len=6,
    causal=False,
    bos="BOS",
    pad="PAD",
    mlp_exactness=100,
    rel_tol=1e-4,
    abs_tol=1e-4,
)
probe_config = ProbeConfig(
    seed=42,
    length=6,
    samples_per_class=64,
    random_candidates=2048,
    structured_candidates=128,
)
OUTPUT_ROOT = PROJECT_ROOT / "results" / "geometry"
FOCUSED_PROGRAMS = (
    "A_prev_token_class",
    "B_histogram_then_repeated_class",
    "P11_check_palindrome",
)
PROGRAMS = build_programs(compiler_config.max_seq_len, include_examples=True)

## Run

 Generate probes, collect activations, check decoded target outputs against RASP,
 and calculate PCA and linear CKA. Each row represents one target token; BOS and PAD
 are excluded. Measurements are saved before plotting, so the manifest's completion
 status describes the numerical study. The final cell reports the figure exports.

In [ ]:
result = run_geometry_study(PROGRAMS, compiler_config, probe_config, OUTPUT_ROOT)
probes = result["probes"]
activations = result["activations"]
analysis = result["analysis"]
RUN_DIR = result["run_dir"]
FIGURE_DIR = RUN_DIR / "figures"
FIGURE_DIR.mkdir()
display(probes.audit)
print(f"Candidate pool: {len(probes.candidates)} unique sequences")
print(f"Saved measurements: {RUN_DIR}")

## Inspect · sampling

 Blue bars show RASP targets; green bars show model outputs. The runner checks their
 agreement numerically. The position audit shows where each class can occur.

In [ ]:
def save_figure(fig, filename):
    fig.savefig(FIGURE_DIR / filename, bbox_inches="tight")
    plt.show()
    plt.close(fig)

def distribution_grid(tables, column, color, filename):
    rows = (len(tables) + 3) // 4
    fig, axes = plt.subplots(rows, 4, figsize=(16, 3 * rows), squeeze=False)
    for ax, (name, table) in zip(axes.flat, tables.items()):
        table[column].value_counts().sort_index().plot.bar(ax=ax, color=color)
        ax.set(title=name, xlabel=column, ylabel="Observations")
        ax.tick_params(axis="x", labelsize=7)
    for ax in list(axes.flat)[len(tables) :]:
        ax.axis("off")
    fig.tight_layout()
    save_figure(fig, filename)


distribution_grid(
    probes.observations,
    "target_class",
    "#4276ab",
    "output_class_balance.png",
)
distribution_grid(
    activations.observations,
    "model_output",
    "#628e67",
    "output_distributions.png",
)

In [ ]:
fig, axes = plt.subplots(
    1, len(FOCUSED_PROGRAMS), figsize=(16, 4), constrained_layout=True
)
for ax, name in zip(np.atleast_1d(axes), FOCUSED_PROGRAMS):
    table = probes.observations[name]
    counts = pd.crosstab(table.target_class, table.position).reindex(
        columns=range(probe_config.length),
        fill_value=0,
    )
    image = ax.imshow(counts.to_numpy(), aspect="auto", cmap="Blues", vmin=0)
    ax.set(
        title=name,
        xticks=range(probe_config.length),
        xlabel="Target position",
        yticks=range(len(counts)),
        yticklabels=counts.index,
        ylabel="Output class",
    )
    fig.colorbar(image, ax=ax, label="Observations")
save_figure(fig, "class_position_balance.png")

## Inspect · dimensions and similarity

 **Full** uses every residual lane. **Computed** excludes token, index, and
 constant-one input lanes. These are views of the same model.

 Participation ratio summarizes the PCA spectrum; `pc95` counts components needed
 to explain 95% of variance. PCA globally centers features without whitening or
 standardization. CKA compares aligned targets across stages within one program;
 gray cells indicate undefined comparisons or absent layers.

In [ ]:
metrics = analysis.metrics
full_layers = metrics[
    (metrics.stage == "Embedding") | metrics.stage.str.endswith("/MLP")
]
display(
    full_layers[full_layers.view == "full"]
    .pivot(
        index="program",
        columns="stage",
        values="participation_ratio",
    )
    .round(3)
)

In [ ]:
def cka_image(ax, matrix, labels, title):
    cmap = plt.get_cmap("viridis").copy()
    cmap.set_bad("#dedede")
    image = ax.imshow(np.ma.masked_invalid(matrix), vmin=0, vmax=1, cmap=cmap)
    ax.set(
        xticks=range(len(labels)),
        yticks=range(len(labels)),
        xticklabels=labels,
        yticklabels=labels,
        title=title,
    )
    ax.tick_params(axis="x", rotation=90, labelsize=7)
    ax.tick_params(axis="y", labelsize=7)
    return image


def dashboard(data, profiles, program, view="full", stage=None, color="model_output"):
    profile = profiles[program][view]
    stages = list(profile["pca"])
    stage = stage or stages[-1]
    fig, axes = plt.subplots(2, 3, figsize=(18, 10), constrained_layout=True)
    for label, line_color in zip(
        stages, plt.cm.viridis(np.linspace(0.1, 0.9, len(stages)))
    ):
        ratio = profile["pca"][label]["explained_variance_ratio"]
        pc = np.arange(1, len(ratio) + 1)
        positive = ratio > 0
        axes[0, 0].semilogy(
            pc[positive], ratio[positive], color=line_color, label=label
        )
        axes[0, 1].plot(pc, np.cumsum(ratio), color=line_color)
    axes[0, 0].set(
        title="PCA spectrum", xlabel="Principal component", ylabel="Variance fraction"
    )
    axes[0, 0].legend(fontsize=7, ncol=2)
    axes[0, 1].axhline(0.95, color="gray", ls="--")
    axes[0, 1].set(
        title="Cumulative variance",
        xlabel="Components",
        ylabel="Variance fraction",
        ylim=(0, 1.03),
    )
    for field, marker, label in (
        ("participation_ratio", "o-", "Participation ratio"),
        ("pc95", "s--", "PCs for 95% variance"),
    ):
        axes[0, 2].plot(
            stages,
            [profile["pca"][label][field] for label in stages],
            marker,
            label=label,
        )
    axes[0, 2].set(title="Effective dimensionality", ylabel="Dimensions")
    axes[0, 2].tick_params(axis="x", rotation=60, labelsize=8)
    axes[0, 2].legend(fontsize=8)
    image = cka_image(
        axes[1, 0], profile["cka"], stages, "Linear CKA: every stage pair"
    )
    fig.colorbar(image, ax=axes[1, 0], fraction=0.046)
    spectra = np.stack(
        [profile["pca"][label]["explained_variance_ratio"] for label in stages]
    )
    image = axes[1, 1].imshow(spectra, aspect="auto", cmap="magma")
    axes[1, 1].set(
        title="Spectrum across stages",
        xlabel="PC index (zero based)",
        yticks=range(len(stages)),
        yticklabels=stages,
    )
    fig.colorbar(image, ax=axes[1, 1], fraction=0.046, label="Variance fraction")
    selected = profile["pca"][stage]
    ax = axes[1, 2]
    if selected["total_variance"] == 0:
        ax.text(
            0.5,
            0.5,
            "This stage has zero variance",
            ha="center",
            transform=ax.transAxes,
        )
        ax.set(title=f"PCA scatter: {stage}")
    else:
        points = pca_coordinates(representation(data, program, view, stage), selected)
        codes, categories = pd.factorize(data.observations[program][color], sort=True)
        image = ax.scatter(
            points[:, 0], points[:, 1], c=codes, cmap="tab20", s=12, alpha=0.55
        )
        bar = fig.colorbar(image, ax=ax, fraction=0.046, ticks=range(len(categories)))
        bar.ax.set_yticklabels([str(value) for value in categories])
        bar.set_label(color)
        ratio = np.pad(
            selected["explained_variance_ratio"],
            (0, max(0, 2 - len(selected["eigenvalues"]))),
        )
        ax.set(
            title=f"PCA scatter: {stage}",
            xlabel=f"PC1 ({ratio[0]:.1%})",
            ylabel=f"PC2 ({ratio[1]:.1%})",
        )
    fig.suptitle(
        f"{program} | {view} | {len(data.observations[program])} balanced targets",
        fontsize=15,
    )
    return fig

In [ ]:
max_layers = max(model.model_config.num_layers for model in activations.models.values())
layer_labels = ["Embedding"] + [f"L{index}/MLP" for index in range(1, max_layers + 1)]
tables = {
    view: full_layers[full_layers.view == view]
    .pivot(
        index="program",
        columns="stage",
        values="participation_ratio",
    )
    .reindex(index=list(PROGRAMS), columns=layer_labels)
    for view in ("full", "computed")
}
fig, axes = plt.subplots(1, 2, figsize=(17, 9), constrained_layout=True)
cmap = plt.get_cmap("magma").copy()
cmap.set_bad("#dedede")
vmax = max(np.nanmax(table.to_numpy()) for table in tables.values())
for ax, (view, table) in zip(axes, tables.items()):
    image = ax.imshow(
        np.ma.masked_invalid(table.to_numpy()),
        aspect="auto",
        cmap=cmap,
        vmin=0,
        vmax=vmax,
    )
    ax.set(
        title=f"Participation ratio: {view}",
        xticks=range(len(layer_labels)),
        xticklabels=layer_labels,
        yticks=range(len(PROGRAMS)),
        yticklabels=list(PROGRAMS),
    )
    ax.tick_params(axis="x", rotation=45)
fig.colorbar(image, ax=axes, label="Effective dimensions", shrink=0.8)
save_figure(fig, "participation_ratio_all_programs.png")

In [ ]:
def cka_gallery(profiles, view):
    fig, axes = plt.subplots(
        (len(profiles) + 3) // 4,
        4,
        figsize=(18, 17),
        squeeze=False,
        constrained_layout=True,
    )
    for ax, (name, views) in zip(axes.flat, profiles.items()):
        image = cka_image(ax, views[view]["cka"], list(views[view]["pca"]), name)
    for ax in list(axes.flat)[len(profiles) :]:
        ax.axis("off")
    fig.colorbar(image, ax=list(axes.flat), shrink=0.5, label="Linear CKA")
    fig.suptitle(f"Every stage-pair CKA: {view}; gray = undefined", fontsize=16)
    save_figure(fig, f"cka_gallery_{view}.png")


cka_gallery(analysis.profiles, "full")
cka_gallery(analysis.profiles, "computed")

## Inspect · a circuit

 Choose a program, residual view, stage, and coloring. The stage options update when
 the program changes. Each dashboard shows spectra, dimensions, stage similarity,
 and a two-dimensional PCA projection.

In [ ]:
program_widget = widgets.Dropdown(
    options=list(PROGRAMS),
    value="B_histogram_then_repeated_class",
    description="Program:",
    layout=widgets.Layout(width="450px"),
)
view_widget = widgets.Dropdown(options=["full", "computed"], description="View:")
stage_widget = widgets.Dropdown(
    options=list(activations.stages[program_widget.value]), description="Stage:"
)
stage_widget.value = stage_widget.options[-1]
color_widget = widgets.Dropdown(
    options=["model_output", "token", "position", "family"], description="Color:"
)


def update_stages(change):
    stage_widget.options = list(activations.stages[change["new"]])
    stage_widget.value = stage_widget.options[-1]


def show_dashboard(program, view, stage, color):
    fig = dashboard(activations, analysis.profiles, program, view, stage, color)
    plt.show()
    plt.close(fig)


program_widget.observe(update_stages, names="value")
output = widgets.interactive_output(
    show_dashboard,
    {
        "program": program_widget,
        "view": view_widget,
        "stage": stage_widget,
        "color": color_widget,
    },
)
display(
    widgets.VBox(
        [
            widgets.HBox([program_widget, view_widget]),
            widgets.HBox([stage_widget, color_widget]),
            output,
        ]
    )
)

## Inspect · individual lanes and trajectories

 Lane variance is measured across sampled targets. A trajectory follows one target
 through every stage in a common PCA coordinate system fitted to all stages.

In [ ]:
def plot_lane_variance(data, program, view="computed", top=25):
    matrix, stages, labels = lane_variances(data, program, view, top)
    fig, ax = plt.subplots(figsize=(13, max(5, 0.28 * len(labels))))
    image = ax.imshow(matrix, aspect="auto", cmap="magma")
    ax.set(
        xticks=range(len(stages)),
        xticklabels=stages,
        yticks=range(len(labels)),
        yticklabels=labels,
        title=f"{program} | {view} | most variable lanes",
    )
    ax.tick_params(axis="x", rotation=45)
    fig.colorbar(image, ax=ax, label="Sample variance")
    fig.tight_layout()
    save_figure(fig, f"lane_variance_{program}_{view}.png")


def plot_trajectory(data, program, probe_id=0, view="full"):
    xy, stages, common, target = trajectory_coordinates(data, program, probe_id, view)
    fig, ax = plt.subplots(figsize=(10, 7))
    (line,) = ax.plot(xy[:, 0], xy[:, 1], "o-", alpha=0.7, label="Selected target")
    ax.scatter(*xy[0], marker="s", s=80, color=line.get_color())
    ax.scatter(*xy[-1], marker="*", s=150, color=line.get_color())
    for index, stage in enumerate(stages):
        ax.annotate(
            stage, xy[index], fontsize=8, xytext=(5, 5), textcoords="offset points"
        )
    ratio = np.pad(
        common["explained_variance_ratio"], (0, max(0, 2 - len(common["eigenvalues"])))
    )
    ax.set(
        title=f"{program} | {view} | probe {probe_id}: {target.tokens}"
        f" | position {target.position} | class {target.target_class}; square = embedding, star = final",
        xlabel=f"Shared PC1 ({ratio[0]:.1%})",
        ylabel=f"Shared PC2 ({ratio[1]:.1%})",
    )
    ax.legend(fontsize=8)
    fig.tight_layout()
    save_figure(fig, f"trajectories_{program}_{view}_probe{probe_id}.png")

In [ ]:
for name in ("B_histogram_then_repeated_class", "P11_check_palindrome"):
    plot_lane_variance(activations, name)
    plot_trajectory(activations, name)

## Interpret

 These measurements describe one compiler revision, vocabulary, and seeded sampling
 design. Different programs have different target distributions; CKA compares stages
 within a program. Repeated targets are weighted observations, not independent trials.

 A zero-variance representation can contain constant computed values. Its PCA spectrum
 and participation ratio are recorded as zero; its CKA comparisons are undefined.
 Changes in effective dimension alone do not establish loss or gain of information.

 Save the focused dashboards alongside the measurements and other figures:

In [ ]:
for name in FOCUSED_PROGRAMS:
    for view in ("full", "computed"):
        fig = dashboard(activations, analysis.profiles, name, view)
        fig.savefig(FIGURE_DIR / f"dashboard_{name}_{view}.png", bbox_inches="tight")
        plt.close(fig)
print(f"Saved {len(list(FIGURE_DIR.glob('*.png')))} figures to {FIGURE_DIR}")

**Next:** [04 · Check alignment with known computation](04_alignment.ipynb).
